In [ ]:
%pip install pandas numpy scikit-learn xgboost matplotlib seaborn --quiet

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Must run from project root (where short_term/ and long_term/ folders live)
if not (Path('short_term').exists() and Path('long_term').exists()):
    raise RuntimeError("Run from the project root directory.")
print("Working directory:", Path('.').resolve())

import new_code_short_term_delu as delu_mod
import new_code_short_term_es   as es_mod
from long_term_delu import LongTermForecasterDELU
from long_term_es   import LongTermForecasterES

plt.rcParams.update({'figure.figsize': (14, 5), 'axes.grid': True, 'grid.alpha': 0.3})
DELU_C, ES_C = '#2196F3', '#FF9800'

# Team 20_Metri — Electricity Price Forecasting

## 1. Methodology Overview

### Dual-regime architecture

The forecasting system automatically selects the appropriate model based on how far
ahead the requested timestamp is, with a cutoff at **14 days**:

| Regime | Horizon | Model | Key inputs |
|--------|---------|-------|-----------|
| Short-term | 0 – 14 days | XGBoost quantile regression | Weather, TSO forecast, gas price, recent price level |
| Long-term  | > 14 days   | Linear regression + climatology shaping | Renewable share, EU ETS carbon, TTF gas scenarios |

---

### Short-term model
Three separate XGBoost regressors (one per quantile: p2.5, p50, p97.5) trained on
~2.5 years of hourly data (2024–2026).

**Key features:**
- **Same-hour level** — exponentially-weighted average of the same UTC hour over the
  past 21 days, restricted to matching day-type (weekday/weekend). Acts as a
  short-term price anchor capturing the current market regime.
- **Weather** — temperature, wind speed, solar radiation, cloud cover, precipitation
  (Open-Meteo historical + forecast)
- **TSO day-ahead forecast** — ENTSO-E scheduled wind+solar generation
- **TTF gas price** — marginal cost anchor
- **Calendar** — hour of day, month, day-of-week

**Validation:** last 60 days held out; cross-validated MAE reported per quantile.

---

### Long-term model
Linear regression trained on monthly (year, month, is_weekend) aggregates from
energy-charts data (DE-LU: 2018–2026, ES: 2017–2026).

**Structural features:** renewable share (%), EU ETS carbon price (EUR/t),
TTF gas (EUR/MWh), month, is_weekend.

**Future inputs** come from a scenario file that linearly interpolates between:
- DE: Energiewende targets — 80% renewable by 2030, 100% by 2035
- ES: PNIEC targets — 81% by 2030, 98% by 2045
- Carbon: EU ETS trajectory — 125 EUR/t by 2030, 200 EUR/t by 2040
- Gas: IEA moderate scenario — gradual decline to 15 EUR/MWh by 2045

**Hourly shaping (climatology):** the regression predicts one average price per
month. To convert to hourly resolution, we multiply by a historical ratio — the
typical price at that (month, hour, weekday/weekend) relative to the monthly mean.
This ratio is computed from 2018–2026 data and captures the daily cycle without
requiring the structural model to learn it. See Section 5 for a full explanation
and visualisation.

**Confidence intervals:** ±20% of forecast at 14-day boundary, growing to ±60%
at 20 years via a log-scale multiplier.

**Physical floor:** `0.3 × gas + 0.15 × carbon` prevents unrealistically low prices
by anchoring to a proxy for the gas CCGT marginal cost.

---

### Why DE-LU and ES differ
| | DE-LU | ES |
|--|-------|-----|
| Dominant renewable | Wind (onshore + offshore) | Solar PV |
| Negative prices | Frequent (high-wind weekends) | Less common |
| Interconnection | Highly connected | Relatively isolated |
| Demand seasonality | Heating-driven winter peaks | Cooling-driven summer peaks |
| Long-term ren. share | 61% (2024) → 100% (2035 target) | 73% (2024) → 98% (2045 target) |

The short-term models learn these differences from data — wind speed dominates
DE-LU feature importance; solar radiation and time-of-day dominate ES.

## 2. Data Loading and Preprocessing

In [ ]:
# Short-term operational data (2024-2026)
print('Loading short-term data...')
prices_delu, gen_delu, tso_delu, weather_delu, gas = delu_mod.load_data()
prices_es,   gen_es,   tso_es,   weather_es,   _   = es_mod.load_data()
print(f"DE-LU  {len(prices_delu):,} hourly rows  "
      f"{prices_delu.index.min().date()} to {prices_delu.index.max().date()}")
print(f"ES     {len(prices_es):,} hourly rows  "
      f"{prices_es.index.min().date()} to {prices_es.index.max().date()}")

In [ ]:
# Long-term historical data — read raw energy-charts CSVs for EDA
def read_ec_prices(folder, skip_disclaimer=False):
    frames = []
    for f in sorted(Path(folder).glob('*.csv')):
        year = next((y for y in range(2017, 2027) if str(y) in f.name), None)
        if year is None:
            continue
        skiprows = [0, 2] if skip_disclaimer else [1]
        df = pd.read_csv(f, skiprows=skiprows)
        date_col  = df.columns[0]
        price_col = next((c for c in df.columns if 'Day Ahead' in c), None)
        if price_col is None:
            continue
        ts = pd.to_datetime(df[date_col], utc=True, errors='coerce')
        s  = pd.to_numeric(df[price_col], errors='coerce')
        frames.append(pd.Series(s.values, index=ts.values))
    out = pd.concat(frames).sort_index()
    out = out[pd.notna(out.index) & ~out.index.duplicated(keep='first')]
    return out.resample('h').mean()

print('Loading long-term historical prices...')
lt_delu_px = read_ec_prices('long_term/input/electricity_prices_delu')
lt_es_px   = read_ec_prices('long_term/input/electricity_prices_es',
                             skip_disclaimer=True)
print(f"DE-LU  {len(lt_delu_px):,} rows  "
      f"{lt_delu_px.index.min().date()} to {lt_delu_px.index.max().date()}")
print(f"ES     {len(lt_es_px):,} rows  "
      f"{lt_es_px.index.min().date()} to {lt_es_px.index.max().date()}")

## 3. Feature Selection and EDA — Side-by-side Zone Comparison

In [ ]:
# Monthly average price — full historical record
fig, ax = plt.subplots(figsize=(14, 5))
lt_delu_px.resample('ME').mean().plot(ax=ax, color=DELU_C, label='DE-LU', linewidth=1.8)
lt_es_px.resample('ME').mean().plot(ax=ax, color=ES_C, label='ES', linewidth=1.8)
ax.set_title('Monthly average Day-Ahead price — DE-LU vs ES')
ax.set_ylabel('EUR/MWh')
ax.legend()
plt.tight_layout()
plt.show()
print("The 2021-2022 energy crisis is visible in both zones. "
      "ES is systematically higher than DE-LU in 2021-2022 "
      "but more moderate in other periods.")

In [ ]:
# Average hourly price profile — winter vs summer, side by side
def hourly_profile(prices, ax, name, color):
    h = prices.dropna().to_frame('p')
    h['hour']   = h.index.hour
    h['month']  = h.index.month
    summer = h[h['month'].isin([6, 7, 8])].groupby('hour')['p'].mean()
    winter = h[h['month'].isin([12, 1, 2])].groupby('hour')['p'].mean()
    ax.plot(summer.index, summer.values, color=color, linewidth=2,
            label=f'{name} Summer')
    ax.plot(winter.index, winter.values, color=color, linewidth=2,
            linestyle='--', label=f'{name} Winter')
    ax.set_xlabel('Hour UTC')
    ax.set_ylabel('EUR/MWh')
    ax.legend(fontsize=9)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
hourly_profile(lt_delu_px, axes[0], 'DE-LU', DELU_C)
hourly_profile(lt_es_px,   axes[1], 'ES',    ES_C)
axes[0].set_title('DE-LU — hourly profile by season')
axes[1].set_title('ES — hourly profile by season')
plt.suptitle('Average price by hour of day (2017–2026)')
plt.tight_layout()
plt.show()
print("ES shows a pronounced solar dip in summer (10:00-16:00 UTC = noon-18:00 local).")
print("DE-LU has a flatter daytime profile — wind generation is less time-of-day dependent.")

In [ ]:
# Annual price distribution (box per year, no outliers shown)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, prices, name, color in [
    (axes[0], lt_delu_px, 'DE-LU', DELU_C),
    (axes[1], lt_es_px,   'ES',    ES_C)
]:
    years = sorted(prices.index.year.unique())
    data  = [prices[prices.index.year == y].dropna().values for y in years]
    bp = ax.boxplot(data, labels=years, patch_artist=True, showfliers=False,
                    medianprops={'color': 'black', 'linewidth': 2})
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(f'{name} — annual price distribution')
    ax.set_ylabel('EUR/MWh')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('Annual price distribution — whiskers show 5th/95th percentile')
plt.tight_layout()
plt.show()
print("DE-LU has more frequent negative prices (high-wind weekends).")
print("Both zones show similar 2022 crisis spike; ES recovers to higher baseline.")

## 4. Feature Engineering

The **same feature vocabulary** is applied to both zones — only the learned weights differ.

### Short-term features (both zones)
| Feature | Source | Notes |
|---------|--------|-------|
| `same_hour_level` | Historical prices | EW avg, last 21 days, matched day-type |
| `wind_speed` | Open-Meteo | Key driver for DE-LU |
| `solar_radiation` | Open-Meteo | Key driver for ES |
| `temperature` | Open-Meteo | Heating/cooling demand |
| `cloud_cover` | Open-Meteo | Solar proxy |
| `precipitation` | Open-Meteo | Hydro proxy (ES) |
| `tso_forecast_mw` | ENTSO-E | Day-ahead wind+solar generation |
| `gas_price` | TTF daily | Marginal cost anchor |
| `hour`, `month`, `dayofweek` | Calendar | Structural patterns |

### Long-term features (both zones)
| Feature | Historical source | Future source |
|---------|-------------------|---------------|
| `renewable_share_pct` | energy-charts generation data | National NECPs + Energiewende/PNIEC |
| `carbon_eur_t` | energy-charts CO2 price | EU ETS published trajectory |
| `gas_eur_mwh` | energy-charts gas price | IEA moderate scenario |
| `month` | — | Calendar |
| `is_weekend` | — | Calendar |

In [ ]:
print('Short-term feature columns:', delu_mod.FEATURE_COLS)
print()
scen = pd.read_csv('long_term/input/scenarios/DE_ES_scenarios.csv')
key_years = scen[scen['year'].isin([2026, 2030, 2035, 2040, 2045])]
print('Long-term scenario inputs:')
print(key_years.to_string(index=False))

## 5. Model Training and Validation

In [ ]:
print('=== Short-term: DE-LU ===')
st_models_delu, sg_delu, st_delu, sw_delu = delu_mod.train(
    prices_delu, gen_delu, tso_delu, weather_delu, gas)

print()
print('=== Short-term: ES ===')
st_models_es, sg_es, st_es, sw_es = es_mod.train(
    prices_es, gen_es, tso_es, weather_es, gas)

In [ ]:
print('=== Long-term models ===')
lt_delu = LongTermForecasterDELU()
lt_es   = LongTermForecasterES()

### How climatology shaping works

The long-term linear regression predicts a **single average price per month** —
for example, "July 2030 average = 40 EUR/MWh". But the submission requires
**hourly** prices. A flat 40 EUR/MWh for every hour of July would be wrong:
solar noon is much cheaper than the evening peak.

**Climatology shaping** solves this by borrowing the hourly *shape* from
historical data, while the long-term model supplies the overall *level*.

From 2018–2026 data, for every combination of (month, hour, weekday/weekend)
we compute:

```
ratio = hourly_price / monthly_mean_price
```

Examples (July, weekday):
- 13:00 UTC → ratio ≈ 0.45  (solar peak suppresses prices)
- 19:00 UTC → ratio ≈ 1.60  (evening demand peak)

Then at forecast time:
```
hourly_p50 = monthly_forecast × ratio[month, hour, day_type]
```

So for a predicted July 2030 monthly average of 40 EUR/MWh:
- 13:00 → 40 × 0.45 = **18 EUR/MWh**
- 19:00 → 40 × 1.60 = **64 EUR/MWh**

The shape of the daily cycle comes from history; only the level shifts
with the structural scenario. The plot below shows the ratio profiles
for both zones across key months.

In [ ]:
# Visualise climatology ratios — how each hour relates to the monthly mean
# lt_delu.climatology and lt_es.climatology are indexed by (month, hour, is_weekend)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=False)
plot_months = [(1, 'January'), (7, 'July')]

for col, (month, month_name) in enumerate(plot_months):
    for row, (lt, name, color) in enumerate([
        (lt_delu, 'DE-LU', DELU_C),
        (lt_es,   'ES',    ES_C),
    ]):
        ax = axes[row][col]
        clim = lt.climatology

        for day_type, ls, label in [(0, '-', 'Weekday'), (1, '--', 'Weekend')]:
            try:
                subset = clim.loc[(month, slice(None), day_type), 'ratio_p50']
                subset.index = subset.index.get_level_values('hour')
                ax.plot(subset.index, subset.values,
                        color=color, linestyle=ls, linewidth=2, label=label)
                # Shade p025-p975 range for weekday only
                if day_type == 0:
                    r025 = clim.loc[(month, slice(None), day_type), 'ratio_p025']
                    r975 = clim.loc[(month, slice(None), day_type), 'ratio_p975']
                    r025.index = r025.index.get_level_values('hour')
                    r975.index = r975.index.get_level_values('hour')
                    ax.fill_between(r025.index, r025.values, r975.values,
                                    alpha=0.15, color=color, label='Weekday p025-p975')
            except KeyError:
                pass

        ax.axhline(1.0, color='black', linewidth=0.8, linestyle=':')
        ax.set_title(f'{name} — {month_name}')
        ax.set_xlabel('Hour UTC')
        ax.set_ylabel('Price / monthly mean')
        ax.legend(fontsize=8)

plt.suptitle(
    'Climatology shape ratios\n'
    'Values above 1.0 = hour is more expensive than the monthly average\n'
    'Values below 1.0 = hour is cheaper (e.g. solar noon in summer)',
    fontsize=10
)
plt.tight_layout()
plt.show()

print("Key observations:")
print("  DE-LU January: strong morning and evening peaks, night trough.")
print("  ES July:       deep midday dip (solar), steep evening recovery.")
print("  The ratio profiles are applied to every long-term monthly forecast")
print("  to reconstruct physically realistic hourly price series.")

In [ ]:
# Validation plot — short-term: predict last VAL_DAYS and compare to actuals
VAL_DAYS  = 60
val_end   = prices_delu.index.max()
val_start = val_end - pd.Timedelta(days=VAL_DAYS)

fc_val_delu = delu_mod.predict(
    val_start.strftime('%Y-%m-%dT%H:%MZ'), val_end.strftime('%Y-%m-%dT%H:%MZ'),
    st_models_delu, prices_delu, gen_delu, tso_delu, weather_delu, gas,
    sg_delu, st_delu, sw_delu)

fc_val_es = es_mod.predict(
    val_start.strftime('%Y-%m-%dT%H:%MZ'), val_end.strftime('%Y-%m-%dT%H:%MZ'),
    st_models_es, prices_es, gen_es, tso_es, weather_es, gas,
    sg_es, st_es, sw_es)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
for ax, fc_v, prices, name, color in [
    (axes[0], fc_val_delu, prices_delu, 'DE-LU', DELU_C),
    (axes[1], fc_val_es,   prices_es,   'ES',    ES_C),
]:
    c025, c50, c975 = fc_v.columns
    actual = prices.reindex(fc_v.index)
    ax.fill_between(fc_v.index, fc_v[c025], fc_v[c975],
                    alpha=0.2, color=color, label='95% interval')
    ax.plot(fc_v.index, fc_v[c50], color=color, linewidth=1.3, label='p50 forecast')
    ax.plot(actual.index, actual.values, color='black', linewidth=0.8,
            alpha=0.7, label='Actual')
    valid = actual.dropna()
    mae = float(np.mean(np.abs(valid - fc_v[c50].reindex(valid.index))))
    ax.set_title(f'{name} — last {VAL_DAYS}-day validation  (p50 MAE = {mae:.2f} EUR/MWh)')
    ax.set_ylabel('EUR/MWh')
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6. Cross-zone Comparison

### Short-term: which features dominate?
DE-LU is **wind-dominated** — wind speed should rank highest.
ES is **solar-dominated** — solar radiation and hour-of-day should rank highest.

### Long-term: what do coefficients say?
The linear regression coefficients show the EUR/MWh change per unit increase in each
structural driver. Both zones share the same marginal fuel (TTF gas), so gas coefficients
should be similar. The renewable share coefficient is negative (more renewables → lower
wholesale price) and stronger for ES where the rapid solar buildout suppresses midday prices.

In [ ]:
# Short-term feature importance — p50 XGBoost model
feat_names = delu_mod.FEATURE_COLS
fi_delu = st_models_delu[0.5].feature_importances_
fi_es   = st_models_es[0.5].feature_importances_

x = np.arange(len(feat_names))
w = 0.35
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w/2, fi_delu, w, label='DE-LU', color=DELU_C, alpha=0.85)
ax.bar(x + w/2, fi_es,   w, label='ES',    color=ES_C,   alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(feat_names, rotation=25, ha='right')
ax.set_ylabel('Feature importance (XGBoost gain)')
ax.set_title('Short-term model — feature importance comparison (p50 model)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Long-term linear regression coefficients
feat_lt   = ['Renewable share', 'Carbon (EUR/t)', 'Gas (EUR/MWh)', 'Month', 'Is weekend']
coef_delu = lt_delu.model.coef_
coef_es   = lt_es.model.coef_

x = np.arange(len(feat_lt))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 0.2, coef_delu, 0.4, label='DE-LU', color=DELU_C, alpha=0.85)
ax.bar(x + 0.2, coef_es,   0.4, label='ES',    color=ES_C,   alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(feat_lt)
ax.set_ylabel('Coefficient (EUR/MWh per unit)')
ax.set_title('Long-term linear regression — structural driver coefficients')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Intercepts:  DE-LU = {lt_delu.model.intercept_:.2f}   ES = {lt_es.model.intercept_:.2f}")

## 7. Long-term Forecast Visualization — Next 2 Years

In [ ]:
now   = pd.Timestamp.now(tz='UTC').floor('h')
start = (now + pd.Timedelta(days=15)).strftime('%Y-%m-%dT%H:%MZ')
end   = (now + pd.Timedelta(days=730)).strftime('%Y-%m-%dT%H:%MZ')

print(f'Forecasting {start} to {end} (daily averages shown)...')
fc_lt_delu = lt_delu.predict(start, end, now)
fc_lt_es   = lt_es.predict(start, end, now)

d_delu = fc_lt_delu.resample('D').mean()
d_es   = fc_lt_es.resample('D').mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
for ax, d, name, color in [
    (axes[0], d_delu, 'DE-LU', DELU_C),
    (axes[1], d_es,   'ES',    ES_C),
]:
    c025, c50, c975 = d.columns
    ax.fill_between(d.index, d[c025], d[c975],
                    alpha=0.2, color=color, label='95% interval')
    ax.plot(d.index, d[c50], color=color, linewidth=1.5, label='p50')
    ax.set_title(f'{name} — structural long-term forecast (daily avg)')
    ax.set_ylabel('EUR/MWh')
    ax.legend()
plt.suptitle('Long-term forecast driven by renewable transition + carbon/gas scenarios')
plt.tight_layout()
plt.show()

## 8. Prediction Generation — Evaluation Window

**Window:** 2026-05-11T02:00 CEST → 2026-05-12T01:00 CEST (= 00:00–23:00 UTC)
Horizon ≈ 2 days ahead → **short-term XGBoost model** activated.

In [ ]:
START_UTC = '2026-05-11T00:00Z'
END_UTC   = '2026-05-11T23:00Z'

fc_delu = delu_mod.predict(START_UTC, END_UTC,
    st_models_delu, prices_delu, gen_delu, tso_delu, weather_delu, gas,
    sg_delu, st_delu, sw_delu)

fc_es = es_mod.predict(START_UTC, END_UTC,
    st_models_es, prices_es, gen_es, tso_es, weather_es, gas,
    sg_es, st_es, sw_es)

fc = pd.concat([fc_delu, fc_es], axis=1)

fig, ax = plt.subplots(figsize=(14, 5))
c025d, c50d, c975d = fc_delu.columns
c025e, c50e, c975e = fc_es.columns
ax.fill_between(fc.index, fc[c025d], fc[c975d], alpha=0.2, color=DELU_C)
ax.plot(fc.index, fc[c50d], color=DELU_C, linewidth=2, label='DE-LU p50')
ax.fill_between(fc.index, fc[c025e], fc[c975e], alpha=0.2, color=ES_C)
ax.plot(fc.index, fc[c50e], color=ES_C, linewidth=2, label='ES p50')
ax.set_title('Evaluation window — 2026-05-11 (UTC hours)')
ax.set_ylabel('EUR/MWh')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save predictions CSV with CEST timestamps
cest = fc.index.tz_convert('Europe/Berlin')
fc.index = [
    ts.strftime('%Y-%m-%dT%H:%M:%S') + ('+02:00' if ts.dst().seconds > 0 else '+01:00')
    for ts in cest
]
fc.index.name = 'timestamp'
fc.to_csv('20_Metri_predictions.csv', float_format='%.2f')
print(f'Saved 20_Metri_predictions.csv  ({len(fc)} rows)')
print(fc.round(2).to_string())